# Path B: Stage 2 Learned Contraction + GINE

Pipeline: Stage 1 (fastloops) -> Stage 2 (learned edge contraction) -> GINE classification -> lift to voxels -> Dice

Stage 2 learns which edges to contract using GT labels at train time, generalizes at test time.

In [ ]:
import numpy as np
import os, re, time, json, gc, warnings
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GINEConv, BatchNorm
from concurrent.futures import ProcessPoolExecutor
import multiprocessing as mp
import fastloops

warnings.filterwarnings('ignore')

DATA_ROOT = "/scratch/ud3d4/acm_data/Data"
RESULTS_DIR = "/home/ud3d4/Desktop/SWOG/results/path_b_stage2"
GRAPH_CACHE = "/dev/shm/path_b_graphs"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(GRAPH_CACHE, exist_ok=True)

HU_MIN, HU_MAX = -50, 250
N_WORKERS = min(mp.cpu_count(), 32)
np.random.seed(42)
torch.manual_seed(42)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"Workers: {N_WORKERS}")
print(f"HU window: [{HU_MIN}, {HU_MAX}]")

## Step 0: Data loading and split

In [ ]:
def load_and_convert(vid):
    ct = np.load(os.path.join(DATA_ROOT, "ct", f"volume-{vid}.npy")).astype(np.float32)
    seg = np.load(os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy")).astype(np.int32)
    ct_u8 = np.clip(ct, HU_MIN, HU_MAX)
    ct_u8 = ((ct_u8 - HU_MIN) / (HU_MAX - HU_MIN) * 255).round().astype(np.uint8)
    ct_u8 = np.ascontiguousarray(ct_u8[..., np.newaxis])  # channel-last
    return ct, seg, ct_u8


def discover_volumes():
    ct_dir = os.path.join(DATA_ROOT, "ct")
    vids = []
    for f in sorted(os.listdir(ct_dir)):
        m = re.match(r"volume-(\d+)\.npy", f)
        if m:
            vid = int(m.group(1))
            seg_path = os.path.join(DATA_ROOT, "seg", f"segmentation-{vid}.npy")
            if os.path.exists(seg_path):
                seg = np.load(seg_path)
                if (seg == 2).sum() > 0:
                    vids.append(vid)
    return sorted(vids)


all_vids = discover_volumes()
print(f"Found {len(all_vids)} LiTS volumes with tumor")

np.random.seed(42)
perm = np.random.permutation(len(all_vids))
n_train = int(0.7 * len(all_vids))
n_val = int(0.15 * len(all_vids))
train_ids = sorted([all_vids[i] for i in perm[:n_train]])
val_ids = sorted([all_vids[i] for i in perm[n_train:n_train + n_val]])
test_ids = sorted([all_vids[i] for i in perm[n_train + n_val:]])
print(f"Split: {len(train_ids)} train / {len(val_ids)} val / {len(test_ids)} test")

## Step 1: Build Stage 1 graphs (fastloops)

Use merge_and_cut with no deletion (keep all supernodes). These are Luke's high-fidelity graphs.

In [ ]:
# Feature extraction from raw node/edge feats (reuse v7 code)
# 3D, C=1 layout: nf=21
# [0]=area, [1-3]=sx,sy,sz, [4-9]=sxx,syy,szz,sxy,sxz,syz,
# [10]=chan0_sum, [11-16]=minx,maxx,miny,maxy,minz,maxz,
# [17]=boundary, [18-20]=canon_z,y,x

def _layout3d(C=1):
    return dict(area=0, s=[1, 2, 3],
                cov=[(4, 0, 0), (5, 1, 1), (6, 2, 2), (7, 0, 1), (8, 0, 2), (9, 1, 2)],
                chan0=10, boundary=10 + C + 6, D=3)


def node_invariants(node_feats, C=1, eps=1e-6):
    """Derive 7 invariant node features from raw moments."""
    f = node_feats.astype(np.float64)
    L = _layout3d(C); D = 3; N = f.shape[0]
    V = f[:, L["area"]]
    Vsafe = np.maximum(V, 1.0)
    mean_coord = np.stack([f[:, c] for c in L["s"]], axis=1) / Vsafe[:, None]
    cov = np.zeros((N, D, D))
    for col, i, j in L["cov"]:
        cij = f[:, col] / Vsafe - mean_coord[:, i] * mean_coord[:, j]
        cov[:, i, j] = cij; cov[:, j, i] = cij
    w = np.linalg.eigvalsh(cov)
    w = np.clip(w, 0.0, None)
    _, vec = np.linalg.eigh(cov)
    principal = vec[..., -1]
    trace = w.sum(axis=1)
    degenerate = trace < eps
    denom = w[:, 2] + eps
    shape = np.stack([(w[:, 2] - w[:, 1]) / denom,
                      (w[:, 1] - w[:, 0]) / denom,
                      w[:, 0] / denom], axis=1)
    shape[degenerate] = 0.0
    line_like = np.where(degenerate, 0.0, shape[:, 0])
    chan = f[:, L["chan0"]:L["chan0"] + C] / Vsafe[:, None] / 255.0
    compactness = f[:, L["boundary"]] / np.power(Vsafe, (D - 1.0) / D)
    elongation = np.where(w[:, 0] > eps, w[:, 2] / (w[:, 0] + eps), 1.0)
    elongation = np.clip(elongation, 1.0, 100.0)
    return dict(V=V, surface=f[:, L["boundary"]], centroid=mean_coord,
                eig=w, shape=shape, line_like=line_like, principal=principal,
                chan=chan, compactness=compactness, elongation=elongation)


def compute_node_features(node_feats, C=1):
    """Build 7-dim node feature matrix."""
    inv = node_invariants(node_feats, C)
    # Canonical principal axis direction
    principal = inv["principal"].astype(np.float32)
    if len(principal):
        max_comp = np.argmax(np.abs(principal), axis=1)
        signs = np.sign(principal[np.arange(len(principal)), max_comp])
        signs[signs == 0] = 1
        principal = principal * signs[:, None]
    x = np.column_stack([
        np.log1p(inv["V"]),
        np.log1p(inv["surface"]),
        inv["compactness"],
        inv["elongation"],
        inv["chan"][:, 0],
        inv["shape"][:, 0],  # planarity
        inv["shape"][:, 2],  # sphericity
    ]).astype(np.float32)
    return x


def compute_edge_features(node_feats, edge_index, edge_feats, C=1, eps=1e-6):
    """Build 12-dim edge feature matrix."""
    inv = node_invariants(node_feats, C, eps)
    a = edge_index[0].astype(np.int64)
    b = edge_index[1].astype(np.int64)
    ef = edge_feats.astype(np.float64)
    blsafe = np.maximum(ef[:, 0], 1.0)
    size_contrast = np.abs(inv["V"][a] - inv["V"][b]) / (inv["V"][a] + inv["V"][b] + eps)
    bfrac_a = ef[:, 0] / (inv["surface"][a] + eps)
    bfrac_b = ef[:, 0] / (inv["surface"][b] + eps)
    mean_contrast = np.abs(inv["chan"][a] - inv["chan"][b])
    shape_dissim = np.abs(inv["shape"][a] - inv["shape"][b])
    axis_align = (np.abs(np.sum(inv["principal"][a] * inv["principal"][b], axis=1))
                  * np.minimum(inv["line_like"][a], inv["line_like"][b]))
    bcontrast = (ef[:, 1] / blsafe) / 255.0
    cut_frac = ef[:, 3] / blsafe
    max_dist_norm = ef[:, 2] / 255.0
    # Log boundary length
    log_bl = np.log1p(ef[:, 0])
    cols = [
        size_contrast[:, None],     # 1
        bfrac_a[:, None],           # 2
        bfrac_b[:, None],           # 3
        mean_contrast if mean_contrast.ndim > 1 else mean_contrast[:, None],  # 4
        shape_dissim,               # 5,6,7
        axis_align[:, None],        # 8
        bcontrast[:, None],         # 9
        cut_frac[:, None],          # 10
        max_dist_norm[:, None],     # 11
        log_bl[:, None],            # 12
    ]
    return np.concatenate(cols, axis=1).astype(np.float32)


def compute_moments_block(node_feats, C=1):
    """Extract the 10 additive moment columns: area, sx,sy,sz, sxx,syy,szz,sxy,sxz,syz."""
    # Indices 0-9 of the raw node_feats
    return node_feats[:, :10].astype(np.float64)


def compute_fg_bg_counts(labels_np, seg):
    """Compute per-supernode fg and bg voxel counts."""
    flat = labels_np.ravel()
    valid = flat >= 0
    gt = (seg.ravel() == 2).astype(np.float64)
    if not valid.any():
        return np.array([]), np.array([])
    max_id = int(flat[valid].max())
    n_fg = np.bincount(flat[valid], weights=gt[valid], minlength=max_id + 1).astype(np.float64)
    total = np.bincount(flat[valid], minlength=max_id + 1).astype(np.float64)
    n_bg = total - n_fg
    return n_fg, n_bg


print("Feature functions defined.")

In [ ]:
# Stage 1 graph params (from v7 search: conservative, high oracle)
PSI = 5   # merge distance
ALPHA = 25  # cut distance


def _build_one_graph(vid):
    """Build Stage 1 graph for one volume. Returns dict with all raw data."""
    import fastloops as fl
    ct_raw, seg, ct_u8 = load_and_convert(vid)
    n_vox = ct_raw.size
    nf, ei, ef, labels, adj = fl.merge_and_cut(
        ct_u8, merge_distance=PSI, cut_distance=ALPHA,
        delete_small_node_max_size=0,
        delete_large_node_min_size=n_vox + 1,
        delete_value_min=0, delete_value_max=255,
        connectivity="faces",
    )
    nf = np.asarray(nf)
    ei = np.asarray(ei)
    ef = np.asarray(ef)
    labels_np = np.asarray(labels)
    return {
        "vid": vid,
        "raw_nf": nf, "raw_ei": ei, "raw_ef": ef,
        "labels": labels_np, "seg": seg, "ct_u8": ct_u8,
    }


# Build all graphs in parallel
all_vids_sorted = sorted(set(train_ids + val_ids + test_ids))
print(f"Building {len(all_vids_sorted)} Stage 1 graphs (PSI={PSI}, ALPHA={ALPHA})...")
t0 = time.time()

stage1_data = {}
with ProcessPoolExecutor(max_workers=N_WORKERS) as pool:
    futures = {pool.submit(_build_one_graph, vid): vid for vid in all_vids_sorted}
    for fut in futures:
        vid = futures[fut]
        try:
            result = fut.result()
            stage1_data[vid] = result
            n_nodes = result["raw_nf"].shape[0]
            n_edges = result["raw_ei"].shape[1]
            split = "train" if vid in train_ids else ("val" if vid in val_ids else "test")
            print(f"  vol-{vid} [{split}]: {n_nodes:,} nodes, {n_edges:,} edges")
        except Exception as e:
            print(f"  vol-{vid}: FAILED - {e}")

print(f"\nBuilt {len(stage1_data)} graphs in {time.time()-t0:.1f}s")

# Quick oracle check
for vid in list(stage1_data.keys())[:3]:
    d = stage1_data[vid]
    flat = d["labels"].ravel(); valid = flat >= 0
    gt = (d["seg"].ravel() == 2).astype(np.float64)
    if valid.any() and gt.sum() > 0:
        max_id = int(flat[valid].max())
        tc = np.bincount(flat[valid], weights=gt[valid], minlength=max_id+1)
        total_c = np.bincount(flat[valid], minlength=max_id+1)
        overlap = tc / np.maximum(total_c, 1)
        tumor_sids = np.where(overlap > 0.10)[0]
        lut = np.zeros(max_id+1, dtype=np.int32); lut[tumor_sids] = 1
        pred = np.where(valid, lut[flat], 0).reshape(d["labels"].shape).astype(bool)
        gm = d["seg"] == 2
        inter = int((pred & gm).sum())
        dice = 2.0 * inter / (pred.sum() + gm.sum() + 1e-8)
        print(f"  vol-{vid} oracle@0.10 = {dice:.4f}")

## Step 2: Stage 2 -- Learned Edge Contraction

A 2-layer GINE learns to score each edge: contract (1) or don't (0).

Training target: edge (u,v) = 1 if both endpoints have same majority label (both fg-majority or both bg-majority).

In [ ]:
# ---------- Stage 2 Edge Scorer ----------

class EdgeScorer(nn.Module):
    """2-layer GINE encoder + edge scoring MLP."""
    def __init__(self, node_dim, edge_dim, hidden=64):
        super().__init__()
        self.edge_proj = nn.Linear(edge_dim, hidden)
        def mlp(d_in):
            return nn.Sequential(
                nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
                nn.Linear(hidden, hidden))
        self.conv1 = GINEConv(mlp(node_dim), edge_dim=hidden)
        self.bn1 = BatchNorm(hidden)
        self.conv2 = GINEConv(mlp(hidden), edge_dim=hidden)
        self.bn2 = BatchNorm(hidden)
        # Edge scoring head: concat(h_u, h_v, edge_feat) -> score
        self.edge_head = nn.Sequential(
            nn.Linear(hidden * 2 + edge_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x, edge_index, edge_attr):
        ea_proj = self.edge_proj(edge_attr)
        h = F.relu(self.bn1(self.conv1(x, edge_index, ea_proj)))
        h = F.relu(self.bn2(self.conv2(h, edge_index, ea_proj)))
        # Score each edge
        src, dst = edge_index[0], edge_index[1]
        edge_repr = torch.cat([h[src], h[dst], edge_attr], dim=1)
        score = self.edge_head(edge_repr).squeeze(-1)
        return score  # raw logits, apply sigmoid for probability


print("EdgeScorer defined.")

In [ ]:
# ---------- Build Stage 2 training data ----------
# For each Stage 1 graph, compute:
# - Node features (7-dim)
# - Edge features (12-dim) -- undirected, so we double edges
# - Edge targets: 1 if both endpoints have same majority label
# - Also store moments and fg/bg for contraction

def build_stage2_data(d):
    """Build PyG Data for edge scoring from Stage 1 graph dict."""
    raw_nf = d["raw_nf"]
    raw_ei = d["raw_ei"]
    raw_ef = d["raw_ef"]
    labels_np = d["labels"]
    seg = d["seg"]
    n_nodes = raw_nf.shape[0]
    n_edges_raw = raw_ei.shape[1]

    # Node features
    x = compute_node_features(raw_nf)
    # Normalize per-graph
    for col in range(x.shape[1]):
        mu, sigma = float(x[:, col].mean()), float(x[:, col].std())
        if sigma > 1e-8:
            x[:, col] = (x[:, col] - mu) / sigma
        else:
            x[:, col] = 0.0

    # Edge features (undirected: double)
    if n_edges_raw > 0:
        ea = compute_edge_features(raw_nf, raw_ei, raw_ef)
        ei_fwd = torch.tensor(raw_ei, dtype=torch.long)
        ei_rev = torch.stack([ei_fwd[1], ei_fwd[0]])
        edge_index = torch.cat([ei_fwd, ei_rev], dim=1)
        edge_attr = torch.tensor(np.concatenate([ea, ea]), dtype=torch.float32)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 12), dtype=torch.float32)

    # Edge targets: majority label agreement
    n_fg, n_bg = compute_fg_bg_counts(labels_np, seg)
    # Majority: 1 = fg-majority, 0 = bg-majority
    if len(n_fg) > 0:
        majority = (n_fg[:n_nodes] > n_bg[:n_nodes]).astype(np.int64)
    else:
        majority = np.zeros(n_nodes, dtype=np.int64)

    if edge_index.shape[1] > 0:
        src = edge_index[0].numpy()
        dst = edge_index[1].numpy()
        # Target: 1 if same majority, 0 if different
        edge_target = (majority[src] == majority[dst]).astype(np.float32)
    else:
        edge_target = np.array([], dtype=np.float32)

    # Moments block for contraction pooling
    moments = compute_moments_block(raw_nf)

    data = Data(
        x=torch.tensor(x, dtype=torch.float32),
        edge_index=edge_index,
        edge_attr=edge_attr,
        edge_target=torch.tensor(edge_target, dtype=torch.float32),
        node_majority=torch.tensor(majority, dtype=torch.long),
    )
    return data


# Build Stage 2 data for all volumes
print("Building Stage 2 edge-scoring data...")
t0 = time.time()
s2_data = {}
for vid in all_vids_sorted:
    if vid not in stage1_data:
        continue
    d = stage1_data[vid]
    s2_data[vid] = build_stage2_data(d)
    ne = s2_data[vid].edge_index.shape[1]
    n_contract = int(s2_data[vid].edge_target.sum())
    if vid in list(all_vids_sorted)[:3]:
        print(f"  vol-{vid}: {s2_data[vid].num_nodes:,} nodes, {ne:,} edges, "
              f"{n_contract:,}/{ne:,} contract targets ({100*n_contract/max(ne,1):.1f}%)")

print(f"Built {len(s2_data)} Stage 2 datasets in {time.time()-t0:.1f}s")

In [ ]:
# ---------- Train Edge Scorer ----------

# Use only graphs that fit in GPU memory for training
MAX_EDGES_GPU = 8_000_000

trainable = [v for v in train_ids if v in s2_data and s2_data[v].edge_index.shape[1] <= MAX_EDGES_GPU]
val_usable = [v for v in val_ids if v in s2_data and s2_data[v].edge_index.shape[1] <= MAX_EDGES_GPU]
print(f"Stage 2 trainable: {len(trainable)}/{len(train_ids)}, val: {len(val_usable)}/{len(val_ids)}")

nd = s2_data[trainable[0]].x.shape[1]
ed = s2_data[trainable[0]].edge_attr.shape[1]
print(f"Node dim: {nd}, Edge dim: {ed}")

scorer = EdgeScorer(nd, ed, hidden=64).to(device)
opt = torch.optim.Adam(scorer.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS_S2 = 30
PATIENCE_S2 = 8
best_val_auc = -1.0
best_state_s2 = None
wait_s2 = 0

print(f"\nTraining EdgeScorer for {EPOCHS_S2} epochs...")
for epoch in range(1, EPOCHS_S2 + 1):
    scorer.train()
    epoch_loss = 0.0
    processed = 0
    for vid in np.random.permutation(trainable):
        g = s2_data[vid]
        if g.edge_index.shape[1] == 0:
            continue
        try:
            gd = g.to(device)
            opt.zero_grad()
            scores = scorer(gd.x, gd.edge_index, gd.edge_attr)
            loss = F.binary_cross_entropy_with_logits(scores, gd.edge_target)
            loss.backward()
            opt.step()
            epoch_loss += loss.item()
            processed += 1
            del gd, scores, loss
        except torch.cuda.OutOfMemoryError:
            try: del gd
            except: pass
            torch.cuda.empty_cache()
            continue
        torch.cuda.empty_cache()

    if processed == 0:
        print(f"  Epoch {epoch}: no graphs processed (all OOM)")
        continue
    mean_loss = epoch_loss / processed

    # Validation: compute accuracy and "AUC-like" metric
    if epoch % 2 == 0 or epoch <= 3:
        scorer.eval()
        correct = 0; total = 0; tp = 0; fp = 0; fn = 0
        with torch.no_grad():
            for vid in val_usable:
                g = s2_data[vid]
                if g.edge_index.shape[1] == 0:
                    continue
                try:
                    gd = g.to(device)
                    scores = scorer(gd.x, gd.edge_index, gd.edge_attr)
                    preds = (scores > 0.0).float()  # logit > 0 = prob > 0.5
                    targets = gd.edge_target
                    correct += int((preds == targets).sum())
                    total += len(targets)
                    tp += int(((preds == 1) & (targets == 1)).sum())
                    fp += int(((preds == 1) & (targets == 0)).sum())
                    fn += int(((preds == 0) & (targets == 1)).sum())
                    del gd, scores, preds, targets
                except torch.cuda.OutOfMemoryError:
                    try: del gd
                    except: pass
                    torch.cuda.empty_cache()
                    continue
                torch.cuda.empty_cache()

        acc = correct / max(total, 1)
        prec = tp / max(tp + fp, 1)
        rec = tp / max(tp + fn, 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-8)

        if f1 > best_val_auc:
            best_val_auc = f1
            best_state_s2 = {k: v.detach().cpu().clone() for k, v in scorer.state_dict().items()}
            wait_s2 = 0
            marker = " *"
        else:
            wait_s2 += 1
            marker = ""
        print(f"  Epoch {epoch:3d}  loss={mean_loss:.4f}  acc={acc:.4f}  "
              f"prec={prec:.4f}  rec={rec:.4f}  F1={f1:.4f}{marker}")
        if wait_s2 >= PATIENCE_S2:
            print(f"  Early stop at epoch {epoch}")
            break
    else:
        print(f"  Epoch {epoch:3d}  loss={mean_loss:.4f}  ({processed} vols)")

if best_state_s2:
    scorer.load_state_dict(best_state_s2)
print(f"\nBest Stage 2 val F1: {best_val_auc:.4f}")
torch.save(best_state_s2 or scorer.state_dict(), os.path.join(RESULTS_DIR, "edge_scorer.pt"))

## Step 2b: Contract graphs using learned edge scores

In [ ]:
# ---------- Contraction execution ----------

def contract_graph(stage1_d, scorer_model, device, threshold=0.5, use_gt=False):
    """
    Contract a Stage 1 graph using learned edge scores (or GT for oracle).
    Returns compressed PyG Data with node labels for GINE training.
    """
    raw_nf = stage1_d["raw_nf"]
    raw_ei = stage1_d["raw_ei"]  # (2, E) original directed half
    raw_ef = stage1_d["raw_ef"]
    labels_np = stage1_d["labels"]
    seg = stage1_d["seg"]
    n_nodes = raw_nf.shape[0]
    n_edges_raw = raw_ei.shape[1]

    if n_edges_raw == 0:
        return None

    # Compute fg/bg for targets and final labels
    n_fg, n_bg = compute_fg_bg_counts(labels_np, seg)
    if len(n_fg) < n_nodes:
        n_fg = np.pad(n_fg, (0, n_nodes - len(n_fg)))
        n_bg = np.pad(n_bg, (0, n_nodes - len(n_bg)))
    n_fg = n_fg[:n_nodes]
    n_bg = n_bg[:n_nodes]

    if use_gt:
        # Oracle: contract edges where both endpoints have same majority
        majority = (n_fg > n_bg).astype(np.int64)
        src_raw = raw_ei[0].astype(np.int64)
        dst_raw = raw_ei[1].astype(np.int64)
        contract_mask = (majority[src_raw] == majority[dst_raw])
    else:
        # Use learned scorer
        scorer_model.eval()
        s2_d = build_stage2_data(stage1_d)
        with torch.no_grad():
            try:
                gd = s2_d.to(device)
                scores = scorer_model(gd.x, gd.edge_index, gd.edge_attr)
                probs = torch.sigmoid(scores).cpu().numpy()
                del gd, scores
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                scores = scorer_model.cpu()(s2_d.x, s2_d.edge_index, s2_d.edge_attr)
                probs = torch.sigmoid(scores).numpy()
                scorer_model.to(device)
        # probs is for doubled edges (fwd + rev). Take first half = original direction
        contract_mask = probs[:n_edges_raw] > threshold

    # Connected components on contracted edges -> merged supernodes
    src_raw = raw_ei[0].astype(np.int64)
    dst_raw = raw_ei[1].astype(np.int64)

    # Union-Find
    parent = np.arange(n_nodes)
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    for i in range(n_edges_raw):
        if contract_mask[i]:
            a, b = find(src_raw[i]), find(dst_raw[i])
            if a != b:
                parent[b] = a

    # Relabel
    roots = np.array([find(i) for i in range(n_nodes)])
    unique_roots, inverse = np.unique(roots, return_inverse=True)
    new_n = len(unique_roots)
    mapping = inverse  # old_node -> new_supernode

    # Pool features
    # Moments are additive
    moments = compute_moments_block(raw_nf)  # (N, 10)
    new_moments = np.zeros((new_n, 10), dtype=np.float64)
    # Also pool chan0_sum and boundary
    chan0_sum = raw_nf[:, 10].astype(np.float64)
    boundary_sum = raw_nf[:, 17].astype(np.float64)
    new_chan0 = np.zeros(new_n, dtype=np.float64)
    new_boundary = np.zeros(new_n, dtype=np.float64)
    new_fg = np.zeros(new_n, dtype=np.float64)
    new_bg = np.zeros(new_n, dtype=np.float64)

    for i in range(n_nodes):
        j = mapping[i]
        new_moments[j] += moments[i]
        new_chan0[j] += chan0_sum[i]
        new_boundary[j] += boundary_sum[i]
        new_fg[j] += n_fg[i]
        new_bg[j] += n_bg[i]

    # Rebuild raw_nf-like array for new supernodes (only need cols 0-10,17)
    # Full 21-col format: we fill what we can (moments + chan0 + boundary)
    new_raw_nf = np.zeros((new_n, raw_nf.shape[1]), dtype=np.uint64)
    new_raw_nf[:, :10] = new_moments.astype(np.uint64)
    new_raw_nf[:, 10] = new_chan0.astype(np.uint64)
    new_raw_nf[:, 17] = new_boundary.astype(np.uint64)
    # min/max bboxes: take min of mins, max of maxes
    for col in [11, 13, 15]:  # min cols
        new_raw_nf[:, col] = np.iinfo(np.uint64).max
    for i in range(n_nodes):
        j = mapping[i]
        for col in [11, 13, 15]:  # min cols
            if raw_nf[i, col] < new_raw_nf[j, col]:
                new_raw_nf[j, col] = raw_nf[i, col]
        for col in [12, 14, 16]:  # max cols
            if raw_nf[i, col] > new_raw_nf[j, col]:
                new_raw_nf[j, col] = raw_nf[i, col]

    # Re-derive edges between merged supernodes
    edge_dict = {}  # (a, b) -> aggregated features
    for i in range(n_edges_raw):
        a = mapping[src_raw[i]]
        b = mapping[dst_raw[i]]
        if a == b:
            continue  # contracted edge
        key = (min(a, b), max(a, b))
        if key not in edge_dict:
            edge_dict[key] = np.zeros(raw_ef.shape[1], dtype=np.float64)
        edge_dict[key] += raw_ef[i].astype(np.float64)

    if len(edge_dict) > 0:
        keys = sorted(edge_dict.keys())
        new_ei = np.array([[k[0] for k in keys], [k[1] for k in keys]], dtype=np.int64)
        new_ef = np.array([edge_dict[k] for k in keys], dtype=np.uint64)
    else:
        new_ei = np.zeros((2, 0), dtype=np.int64)
        new_ef = np.zeros((0, raw_ef.shape[1]), dtype=np.uint64)

    # Build node features
    x = compute_node_features(new_raw_nf)
    for col in range(x.shape[1]):
        mu, sigma = float(x[:, col].mean()), float(x[:, col].std())
        if sigma > 1e-8:
            x[:, col] = (x[:, col] - mu) / sigma
        else:
            x[:, col] = 0.0

    # Build edge features
    if new_ei.shape[1] > 0:
        ea = compute_edge_features(new_raw_nf, new_ei, new_ef)
        ei_fwd = torch.tensor(new_ei, dtype=torch.long)
        ei_rev = torch.stack([ei_fwd[1], ei_fwd[0]])
        edge_index = torch.cat([ei_fwd, ei_rev], dim=1)
        edge_attr = torch.tensor(np.concatenate([ea, ea]), dtype=torch.float32)
    else:
        edge_index = torch.zeros((2, 0), dtype=torch.long)
        edge_attr = torch.zeros((0, 12), dtype=torch.float32)

    # Node labels for GINE: fg-majority
    y = (new_fg > new_bg).astype(np.int64)

    # For voxel lifting: build mapping from voxel label -> new supernode
    # mapping[old_node] = new_supernode

    data = Data(
        x=torch.tensor(x, dtype=torch.float32),
        edge_index=edge_index,
        edge_attr=edge_attr,
        y=torch.tensor(y, dtype=torch.long),
    )
    return data, mapping, new_fg, new_bg


print("Contraction function defined.")

In [ ]:
# ---------- Contract all graphs ----------
print("Contracting all graphs with learned edge scorer...")
t0 = time.time()

compressed_data = {}  # vid -> (Data, mapping, n_fg, n_bg)
compression_stats = []

for vid in all_vids_sorted:
    if vid not in stage1_data:
        continue
    d = stage1_data[vid]
    is_train = vid in train_ids
    # For train/val: use GT contraction to get best training signal for GINE
    # For test: use learned scorer
    use_gt = (vid in train_ids or vid in val_ids)
    result = contract_graph(d, scorer, device, threshold=0.5, use_gt=use_gt)
    if result is None:
        print(f"  vol-{vid}: skip (no edges)")
        continue
    cdata, mapping, c_fg, c_bg = result
    compressed_data[vid] = (cdata, mapping, c_fg, c_bg)
    old_n = d["raw_nf"].shape[0]
    new_n = cdata.num_nodes
    ratio = old_n / max(new_n, 1)
    n_tu = int((cdata.y == 1).sum())
    split = "train" if vid in train_ids else ("val" if vid in val_ids else "test")
    compression_stats.append({"vid": vid, "split": split, "old": old_n, "new": new_n, "ratio": ratio})
    if vid in list(all_vids_sorted)[:5] or vid in test_ids[:3]:
        print(f"  vol-{vid} [{split}]: {old_n:,} -> {new_n:,} nodes ({ratio:.1f}x), "
              f"{cdata.num_edges:,} edges, {n_tu} tumor nodes")

mean_ratio = np.mean([s["ratio"] for s in compression_stats])
mean_new = np.mean([s["new"] for s in compression_stats])
print(f"\nContracted {len(compressed_data)} graphs in {time.time()-t0:.1f}s")
print(f"Mean compression: {mean_ratio:.1f}x, mean nodes: {mean_new:.0f}")

In [ ]:
# ---------- Oracle Dice on compressed graphs ----------
print("\nOracle Dice on compressed graphs:")
oracle_dices = []
for vid in all_vids_sorted[:10]:
    if vid not in compressed_data or vid not in stage1_data:
        continue
    cdata, mapping, c_fg, c_bg = compressed_data[vid]
    d = stage1_data[vid]
    labels_np = d["labels"]
    seg = d["seg"]
    
    # Oracle: label compressed nodes by majority
    new_n = cdata.num_nodes
    oracle_pred = (c_fg > c_bg).astype(np.int64)
    
    # Lift to voxels
    flat = labels_np.ravel()
    valid = flat >= 0
    if not valid.any():
        continue
    max_old = int(flat[valid].max())
    # mapping: old_node -> new_supernode
    lut = np.zeros(max_old + 1, dtype=np.int64)
    for old_id in range(min(len(mapping), max_old + 1)):
        new_id = mapping[old_id]
        lut[old_id] = oracle_pred[new_id]
    pred_vox = np.where(valid, lut[flat], 0).reshape(labels_np.shape).astype(bool)
    gt_vox = seg == 2
    inter = int((pred_vox & gt_vox).sum())
    dice = 2.0 * inter / (pred_vox.sum() + gt_vox.sum() + 1e-8)
    oracle_dices.append(dice)
    split = "train" if vid in train_ids else ("val" if vid in val_ids else "test")
    print(f"  vol-{vid} [{split}]: oracle Dice = {dice:.4f}")

print(f"\nMean oracle Dice (first 10): {np.mean(oracle_dices):.4f}")

## Step 3: Train GINE on compressed graphs

In [ ]:
# ---------- GINE model ----------

class GINE(nn.Module):
    def __init__(self, nd, ed, h=128):
        super().__init__()
        self.ep = nn.Linear(ed, h)
        def mlp(d):
            return nn.Sequential(nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Linear(h, h))
        self.c1 = GINEConv(mlp(nd), edge_dim=h); self.b1 = BatchNorm(h)
        self.c2 = GINEConv(mlp(h), edge_dim=h); self.b2 = BatchNorm(h)
        self.c3 = GINEConv(mlp(h), edge_dim=h); self.b3 = BatchNorm(h)
        self.head = nn.Linear(h, 2)

    def forward(self, x, ei, ea):
        if ea is not None and ea.numel() > 0:
            ea = self.ep(ea)
        else:
            n = x.size(0)
            ei = torch.stack([torch.arange(n, device=x.device)] * 2)
            ea = torch.zeros(n, self.ep.out_features, device=x.device)
        x = F.relu(self.b1(self.c1(x, ei, ea)))
        x = F.relu(self.b2(self.c2(x, ei, ea)))
        x = F.relu(self.b3(self.c3(x, ei, ea)))
        return self.head(x)


# Prepare data
MAX_NODES_GPU = 3_500_000
gine_train = [v for v in train_ids if v in compressed_data and compressed_data[v][0].num_nodes <= MAX_NODES_GPU]
gine_val = [v for v in val_ids if v in compressed_data]
gine_test = [v for v in test_ids if v in compressed_data]
print(f"GINE: {len(gine_train)} train, {len(gine_val)} val, {len(gine_test)} test")

# Class weights
total_pos = sum(int((compressed_data[v][0].y == 1).sum()) for v in gine_train)
total_neg = sum(int((compressed_data[v][0].y == 0).sum()) for v in gine_train)
ratio = total_neg / max(total_pos, 1)
eff = min(np.sqrt(ratio), 30.0)
class_weight = torch.tensor([1.0, eff], dtype=torch.float32).to(device)
print(f"Class weight: [1.0, {eff:.1f}] (ratio: {ratio:.0f}:1)")

nd_gine = compressed_data[gine_train[0]][0].x.shape[1]
ed_gine = compressed_data[gine_train[0]][0].edge_attr.shape[1] if compressed_data[gine_train[0]][0].edge_attr.numel() > 0 else 12
print(f"GINE node dim: {nd_gine}, edge dim: {ed_gine}")

In [ ]:
# ---------- Train GINE ----------

gine_model = GINE(nd_gine, ed_gine).to(device)
gine_opt = torch.optim.Adam(gine_model.parameters(), lr=1e-3, weight_decay=1e-4)

EPOCHS_GINE = 200
PATIENCE_GINE = 15
VAL_EVERY = 3
best_dice_gine = -1.0
best_state_gine = None
wait_gine = 0
history = {"train_loss": [], "val_dice": []}

print(f"Training GINE for up to {EPOCHS_GINE} epochs...")
for epoch in range(1, EPOCHS_GINE + 1):
    gine_model.train()
    epoch_loss = 0.0
    processed = 0
    for vid in np.random.permutation(gine_train):
        g = compressed_data[vid][0]
        try:
            gd = g.to(device)
            gine_opt.zero_grad()
            logits = gine_model(gd.x, gd.edge_index, gd.edge_attr)
            loss = F.cross_entropy(logits, gd.y, weight=class_weight)
            loss.backward()
            gine_opt.step()
            epoch_loss += loss.item()
            processed += 1
            del gd, logits, loss
        except torch.cuda.OutOfMemoryError:
            try: del gd
            except: pass
            torch.cuda.empty_cache()
            continue
        torch.cuda.empty_cache()

    if processed == 0:
        print(f"  Epoch {epoch}: all OOM")
        continue
    mean_loss = epoch_loss / processed
    history["train_loss"].append(mean_loss)

    if epoch % VAL_EVERY == 0 or epoch <= 3:
        gine_model.eval()
        tp = fp = fn = 0
        with torch.no_grad():
            for vid in gine_val:
                cdata, mapping, c_fg, c_bg = compressed_data[vid]
                d = stage1_data[vid]
                labels_np = d["labels"]
                seg_np = d["seg"]
                try:
                    gd = cdata.to(device)
                    preds = gine_model(gd.x, gd.edge_index, gd.edge_attr).argmax(dim=1).cpu().numpy()
                    del gd
                except torch.cuda.OutOfMemoryError:
                    try: del gd
                    except: pass
                    torch.cuda.empty_cache()
                    preds = gine_model.cpu()(cdata.x, cdata.edge_index, cdata.edge_attr).argmax(dim=1).numpy()
                    gine_model.to(device)
                torch.cuda.empty_cache()

                # Lift to voxels
                flat = labels_np.ravel(); valid = flat >= 0
                if not valid.any():
                    continue
                max_old = int(flat[valid].max())
                lut = np.zeros(max_old + 1, dtype=np.int8)
                for old_id in range(min(len(mapping), max_old + 1)):
                    new_id = mapping[old_id]
                    if new_id < len(preds):
                        lut[old_id] = preds[new_id]
                pred_vox = np.where(valid, lut[flat], 0).reshape(labels_np.shape).astype(bool)
                gt_vox = seg_np == 2
                inter = int((pred_vox & gt_vox).sum())
                tp += inter
                fp += int(pred_vox.sum()) - inter
                fn += int(gt_vox.sum()) - inter

        vd = 2 * tp / (2 * tp + fp + fn + 1e-8)
        history["val_dice"].append(vd)

        if vd > best_dice_gine:
            best_dice_gine = vd
            best_state_gine = {k: v.detach().cpu().clone() for k, v in gine_model.state_dict().items()}
            wait_gine = 0
            marker = " *"
        else:
            wait_gine += 1
            marker = ""
        print(f"  Epoch {epoch:3d}  loss={mean_loss:.4f}  val_dice={vd:.4f}  ({processed} vols){marker}")
        if wait_gine >= PATIENCE_GINE:
            print(f"  Early stop at epoch {epoch}, best val Dice={best_dice_gine:.4f}")
            break
    else:
        print(f"  Epoch {epoch:3d}  loss={mean_loss:.4f}  ({processed} vols)")

if best_state_gine:
    gine_model.load_state_dict(best_state_gine)
print(f"\nBest GINE val Dice: {best_dice_gine:.4f}")
torch.save(best_state_gine or gine_model.state_dict(), os.path.join(RESULTS_DIR, "gine_model.pt"))

## Step 4: Full pipeline evaluation

In [ ]:
# ---------- Final evaluation ----------
# For test: use learned edge scorer for contraction (not GT)
# For train/val: use GT contraction (same as training)

print("\n" + "=" * 70)
print("  FULL PIPELINE EVALUATION")
print("=" * 70)

gine_model.eval()
results = []

for vid in sorted(compressed_data.keys()):
    cdata, mapping, c_fg, c_bg = compressed_data[vid]
    d = stage1_data[vid]
    labels_np = d["labels"]
    seg_np = d["seg"]
    split = "train" if vid in train_ids else ("val" if vid in val_ids else "test")

    with torch.no_grad():
        try:
            gd = cdata.to(device)
            preds = gine_model(gd.x, gd.edge_index, gd.edge_attr).argmax(dim=1).cpu().numpy()
            del gd
        except (torch.cuda.OutOfMemoryError, RuntimeError):
            torch.cuda.empty_cache()
            preds = gine_model.cpu()(cdata.x, cdata.edge_index, cdata.edge_attr).argmax(dim=1).numpy()
            gine_model.to(device)
        torch.cuda.empty_cache()

    # Lift to voxels
    flat = labels_np.ravel(); valid = flat >= 0
    pred_vox = np.zeros(labels_np.shape, dtype=bool)
    if valid.any():
        max_old = int(flat[valid].max())
        lut = np.zeros(max_old + 1, dtype=np.int8)
        for old_id in range(min(len(mapping), max_old + 1)):
            new_id = mapping[old_id]
            if new_id < len(preds):
                lut[old_id] = preds[new_id]
        pred_vox = np.where(valid, lut[flat], 0).reshape(labels_np.shape).astype(bool)

    gt_vox = seg_np == 2
    inter = int((pred_vox & gt_vox).sum())
    dice = 2.0 * inter / (pred_vox.sum() + gt_vox.sum() + 1e-8)
    rec = inter / (gt_vox.sum() + 1e-8)
    prec = inter / (pred_vox.sum() + 1e-8) if pred_vox.sum() > 0 else 0.0
    results.append({"vid": vid, "split": split, "dice": float(dice),
                    "recall": float(rec), "precision": float(prec),
                    "nodes": int(cdata.num_nodes)})

# Summary
print("\n" + "=" * 70)
print("  SUMMARY")
print("=" * 70)
for split in ["train", "val", "test"]:
    scores = [r["dice"] for r in results if r["split"] == split]
    recs = [r["recall"] for r in results if r["split"] == split]
    precs = [r["precision"] for r in results if r["split"] == split]
    nodes = [r["nodes"] for r in results if r["split"] == split]
    if scores:
        print(f"  {split:>5s}: Dice = {np.mean(scores):.4f} +/- {np.std(scores):.4f}  "
              f"Recall = {np.mean(recs):.4f}  Prec = {np.mean(precs):.4f}  "
              f"Nodes = {np.mean(nodes):.0f}  (n={len(scores)})")

print(f"\n  Stage 1 params: PSI={PSI} ALPHA={ALPHA} HU=[{HU_MIN},{HU_MAX}]")
print(f"  Stage 2: EdgeScorer val F1 = {best_val_auc:.4f}")
print(f"  GINE best val Dice: {best_dice_gine:.4f}")
print(f"  Paper target: Dice 0.891 +/- 0.007")

# Per-volume test results
print("\nPer-volume test results:")
for r in sorted([r for r in results if r["split"] == "test"], key=lambda x: x["vid"]):
    print(f"  vol-{r['vid']}: Dice={r['dice']:.4f}  Rec={r['recall']:.4f}  "
          f"Prec={r['precision']:.4f}  Nodes={r['nodes']:,}")

In [ ]:
# ---------- Save results ----------
final_results = {
    "params": {
        "psi": PSI, "alpha": ALPHA,
        "hu_min": HU_MIN, "hu_max": HU_MAX,
        "stage2_hidden": 64, "stage2_threshold": 0.5,
        "gine_hidden": 128, "gine_epochs": EPOCHS_GINE,
    },
    "stage2_val_f1": float(best_val_auc),
    "gine_best_val_dice": float(best_dice_gine),
    "compression_stats": compression_stats,
    "history": history,
    "results": results,
}

# Per-split summary
for split in ["train", "val", "test"]:
    scores = [r["dice"] for r in results if r["split"] == split]
    if scores:
        final_results[f"{split}_dice_mean"] = float(np.mean(scores))
        final_results[f"{split}_dice_std"] = float(np.std(scores))

with open(os.path.join(RESULTS_DIR, "results.json"), "w") as f:
    json.dump(final_results, f, indent=2)

print(f"Results saved to {RESULTS_DIR}/results.json")
print("Done.")